# Fe3O4-RWGS最小モデルの温度依存性

このNotebookでは、Fe3O4上RWGSの7ステップ最小モデルを使って、温度に対するCO生成速度と表面被覆率の変化を確認します。

ここで扱う範囲はFe3O4-RWGSのみです。FeCx-FT、C2+生成、相変化を含むモデルは扱いません。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from microkinetics.rates import RATE_CONSTANT_NAMES
from microkinetics.solver import co_production_rate_from_result, solve_steady_state

## 入力条件

温度範囲、気相分圧、速度定数を設定します。速度定数は計算手順を確認するための仮の値であり、実験値や文献値ではありません。

In [ ]:
temperatures = np.linspace(450.0, 700.0, 11)  # K

partial_pressures = {
    "CO2": 0.25,
    "H2": 0.75,
    "CO": 1.0e-6,
    "H2O": 1.0e-6,
}

R_GAS = 8.314462618  # J mol^-1 K^-1


def arrhenius(prefactor, activation_energy_j_mol):
    """Return a temperature-dependent example rate constant."""

    return lambda temperature: prefactor * np.exp(
        -activation_energy_j_mol / (R_GAS * temperature)
    )


rate_constants = {}
for index, name in enumerate(RATE_CONSTANT_NAMES, start=1):
    prefactor = 1.0e2 if name.endswith("f") else 5.0e1
    activation_energy = 20_000.0 + 750.0 * index
    rate_constants[name] = arrhenius(prefactor, activation_energy)

partial_pressures

## 計算

各温度で定常表面被覆率を解き、R4に対応するCO生成速度を取り出します。結果は`pandas.DataFrame`として整理します。

In [ ]:
rows = []

for temperature in temperatures:
    result = solve_steady_state(
        float(temperature),
        partial_pressures=partial_pressures,
        rate_constants=rate_constants,
    )
    if not result.success:
        raise RuntimeError(
            f"Steady-state solve failed at {temperature:.1f} K: {result.message}"
        )

    row = {
        "temperature_K": float(temperature),
        "co_production_rate": co_production_rate_from_result(result),
    }
    row.update({f"theta_{name}": value for name, value in result.coverages.items()})
    rows.append(row)

df = pd.DataFrame(rows)
df

## 可視化

温度に対するCO生成速度をプロットします。単位はここではモデル内の速度定数に依存する任意単位として扱います。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["temperature_K"], df["co_production_rate"], marker="o")
ax.set_xlabel("Temperature / K")
ax.set_ylabel("CO production rate / arbitrary units")
ax.set_title("Fe3O4-RWGS minimal model")
ax.grid(True, alpha=0.3)
fig.tight_layout()

主要な表面被覆率の温度依存性も確認します。

In [ ]:
coverage_columns = [
    "theta_*",
    "theta_H*",
    "theta_CO*",
    "theta_O*",
    "theta_OH*",
    "theta_H2O*",
]

fig, ax = plt.subplots(figsize=(7, 4))
for column in coverage_columns:
    ax.plot(
        df["temperature_K"],
        df[column],
        marker="o",
        label=column.replace("theta_", "theta "),
    )

ax.set_xlabel("Temperature / K")
ax.set_ylabel("Coverage")
ax.set_ylim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fig.tight_layout()

## 考察

この仮パラメータでは、温度上昇に伴ってCO生成速度が増加します。これはArrhenius型の温度依存性を持つ仮の速度定数を使っているためであり、実触媒の活性や見かけ活性化エネルギーを示すものではありません。

温度を変えると、`H*`、`O*`、`OH*`などの被覆率も変化し得ます。RWGSではCO2の解離、H2解離吸着、O*の水素化、H2O脱離が連動するため、ひとつのステップだけではなく、表面中間体全体のバランスとして速度が決まります。

このNotebookの限界は、速度定数が仮の値であること、熱力学整合性を厳密に課したパラメータセットではないこと、被覆率依存性や複数サイトを含めていないことです。FeCx-FTやC2+生成はこのNotebookの範囲外です。

## 演習問題

1. `partial_pressures["CO2"]`を変え、CO生成速度と`CO2*`被覆率がどう変わるか調べよ。
2. `partial_pressures["H2"]`を変え、`H*`、`OH*`、`H2O*`被覆率への影響を調べよ。
3. `partial_pressures["H2O"]`を大きくして、H2O添加がCO生成速度に与える影響を確認せよ。
4. 特定の反応、たとえばR3またはR5の速度定数を変え、CO生成速度の感度を比較せよ。